# 06 · Honest evaluation — the whole point

Rules: subject-level splits, every learned step fit inside the fold, **cross-subject (LOSO)** headline as mean ± std, imbalance-aware metrics + confusion matrix, protocol tagged on every number.

First we *prove* leakage inflates scores, then we report the honest leaderboard.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
%load_ext autoreload
%autoreload 2
import numpy as np, matplotlib.pyplot as plt


In [ ]:
from eeglog.data import load_moabb
from eeglog.models_classic import csp_lda, all_classical
from eeglog import evaluation as E
d = load_moabb(subjects=[1, 2, 3], paradigm='left_right')

### The leakage trap, made concrete

In [ ]:
demo = E.leakage_demo(csp_lda, d.X, d.y, d.groups)
print(f"leaky  CSP-on-all-data : {demo['leaky_mean']:.3f}")
print(f"honest CSP-in-fold     : {demo['honest_mean']:.3f}")
print(f"inflation              : +{demo['inflation']:.3f}")

### Within-subject vs cross-subject (same model)

In [ ]:
m = csp_lda()
within = E.evaluate_within_subject(m, d.X, d.y, d.groups, model_name='CSP+LDA')
loso = E.evaluate_loso(csp_lda(), d.X, d.y, d.groups, model_name='CSP+LDA')
print(within); print(loso)  # cross-subject is lower — and honest

### Honest headline leaderboard (LOSO, mean ± std)

In [ ]:
results = [E.evaluate_loso(mdl, d.X, d.y, d.groups, model_name=n)
           for n, mdl in all_classical(d.sfreq).items()]
lb = E.leaderboard(results); display(lb)

In [ ]:
# Confusion matrix for the top model.
best = all_classical(d.sfreq)[lb.iloc[0]['model']]
cm, labels = E.confusion(best, d.X, d.y, d.groups)
import matplotlib.pyplot as plt
plt.imshow(cm, cmap='Blues'); plt.xticks(range(len(labels)), labels, rotation=45)
plt.yticks(range(len(labels)), labels); plt.title(f"LOSO confusion — {lb.iloc[0]['model']}")
plt.colorbar()

**What would be dishonest here:** quoting the within-subject or leaky number as the result; reporting raw accuracy on imbalanced classes; or fitting CSP/scaler/ICA on the full dataset before CV. The headline is the **LOSO mean ± std** above.